In [1]:
import os
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from PIL import Image
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np
import warnings
from collections import Counter
import zipfile
import base64
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML, Javascript, display

warnings.filterwarnings("ignore")

# 1. SETUP & CONFIGURATION
gc.collect()
torch.cuda.empty_cache()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16       
EPOCHS = 60           
DATA_PATH = "/kaggle/input/cs-776-competition-2025-2026-sem-2/dataset_music"

print(f"✅ Method 1 (Platinum Full Pipeline) running on: {DEVICE}")

# 2. DATASET & TRANSFORMS
class TripleImageDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None):
        self.root_dir = root_dir
        self.split = split
        self.transform = transform
        csv_path = os.path.join(root_dir, "train" if split != "test" else "test", "metadata.csv")
        self.metadata = pd.read_csv(csv_path)
        self.img_dir = os.path.join(root_dir, "train") if split != "test" else os.path.join(root_dir, "test")

    def __len__(self): return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        if self.split == "test":
            fname = f"{int(row['id']):06d}.jpg"
            return self.load_images(fname), row['id']
        else:
            fname = f"{int(row['id']):06d}.jpg"
            return self.load_images(fname), torch.tensor(int(row['target']), dtype=torch.long)

    def load_images(self, fname):
        imgs = []
        for i in range(1, 4):
            try:
                p = os.path.join(self.img_dir, f"input_{i}", fname)
                img = Image.open(p).convert('RGB')
                if self.transform: img = self.transform(img)
                imgs.append(img)
            except:
                imgs.append(torch.zeros(3, 128, 128))
        return imgs[0], imgs[1], imgs[2]

# Strong Augmentation for Training (RandomErasing closes the gap)
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)) 
])

# Clean Transform for Validation/Test
val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 3. ROBUST DATA SPLITTING
full_meta = pd.read_csv(os.path.join(DATA_PATH, "train", "metadata.csv"))
dataset_size = len(full_meta)
indices = list(range(dataset_size))
split = int(np.floor(0.15 * dataset_size)) # 15% Validation
np.random.seed(42)
np.random.shuffle(indices)
train_indices, val_indices = indices[split:], indices[:split]

train_ds = Subset(TripleImageDataset(DATA_PATH, split="train", transform=train_transform), train_indices)
val_ds = Subset(TripleImageDataset(DATA_PATH, split="train", transform=val_transform), val_indices)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"📊 Data Split: {len(train_ds)} Training | {len(val_ds)} Validation")

# 4. PLATINUM ARCHITECTURE (SiLU + SE Attention)
class FastRectConv2D(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, p=1):
        super().__init__()
        self.k = k if isinstance(k, tuple) else (k, k)
        self.s = s if isinstance(s, tuple) else (s, s)
        self.p = p if isinstance(p, tuple) else (p, p)
        self.weight = nn.Parameter(torch.randn(out_c, in_c, self.k[0], self.k[1]) * 0.01)
        self.bias = nn.Parameter(torch.zeros(out_c))

    def forward(self, x):
        B, C, H, W = x.shape
        if self.p[0] > 0 or self.p[1] > 0: 
            x = F.pad(x, (self.p[1], self.p[1], self.p[0], self.p[0]))
        patches = F.unfold(x, kernel_size=self.k, stride=self.s)
        out = (self.weight.view(self.weight.shape[0], -1) @ patches) + self.bias.view(-1, 1)
        h_out = (H + 2*self.p[0] - self.k[0]) // self.s[0] + 1
        w_out = (W + 2*self.p[1] - self.k[1]) // self.s[1] + 1
        return out.view(B, -1, h_out, w_out)

class SEBlock(nn.Module):
    def __init__(self, channel, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.SiLU(),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class DeepFastCNN_Platinum(nn.Module):
    def __init__(self):
        super().__init__()
        # Block 1
        self.conv1_1 = FastRectConv2D(3, 24, k=3, p=1)
        self.bn1_1 = nn.BatchNorm2d(24)
        self.conv1_2 = FastRectConv2D(24, 24, k=3, p=1)
        self.bn1_2 = nn.BatchNorm2d(24)
        self.se1 = SEBlock(24, reduction=4)
        self.pool1 = nn.MaxPool2d(2, 2)
        # Block 2
        self.conv2_1 = FastRectConv2D(24, 48, k=3, p=1)
        self.bn2_1 = nn.BatchNorm2d(48)
        self.conv2_2 = FastRectConv2D(48, 48, k=3, p=1)
        self.bn2_2 = nn.BatchNorm2d(48)
        self.se2 = SEBlock(48, reduction=8)
        self.pool2 = nn.MaxPool2d(2, 2)
        # Block 3
        self.conv3_1 = FastRectConv2D(48, 96, k=3, p=1)
        self.bn3_1 = nn.BatchNorm2d(96)
        self.conv3_2 = FastRectConv2D(96, 96, k=3, p=1)
        self.bn3_2 = nn.BatchNorm2d(96)
        self.se3 = SEBlock(96, reduction=16)
        self.pool3 = nn.MaxPool2d(2, 2)
        
    def forward(self, x):
        x = F.silu(self.bn1_1(self.conv1_1(x)))
        x = self.pool1(self.se1(F.silu(self.bn1_2(self.conv1_2(x)))))
        x = F.silu(self.bn2_1(self.conv2_1(x)))
        x = self.pool2(self.se2(F.silu(self.bn2_2(self.conv2_2(x)))))
        x = F.silu(self.bn3_1(self.conv3_1(x)))
        x = self.pool3(self.se3(F.silu(self.bn3_2(self.conv3_2(x)))))
        return x.mean(dim=(2, 3)) 

class DeepMethod1Model_Platinum(nn.Module):
    def __init__(self):
        super().__init__()
        self.b1 = DeepFastCNN_Platinum()
        self.b2 = DeepFastCNN_Platinum()
        self.b3 = DeepFastCNN_Platinum()
        self.fc = nn.Linear(288, 16) 
        
    def forward(self, x1, x2, x3):
        return self.fc(torch.cat([self.b1(x1), self.b2(x2), self.b3(x3)], dim=1))

# 5. TRAINING LOOP
def train_and_eval():
    model = DeepMethod1Model_Platinum().to(DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"🧠 Platinum Model Parameters: {total_params:,} (Limit: 500,000)")
    
    targets = full_meta["target"].tolist()
    counts = Counter(targets)
    weights = torch.tensor([len(targets)/counts[i] for i in range(16)], dtype=torch.float32).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
    
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=15, T_mult=2)

    best_f1 = 0.0
    print("🚀 Starting Platinum Training...")
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        train_true, train_preds = [], []
        
        for (x1, x2, x3), y in train_loader:
            x1, x2, x3, y = x1.to(DEVICE), x2.to(DEVICE), x3.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x1, x2, x3)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_preds.extend(logits.argmax(1).cpu().numpy())
            train_true.extend(y.cpu().numpy())
            
        scheduler.step()
        train_f1 = f1_score(train_true, train_preds, average='macro')
        
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for (x1, x2, x3), y in val_loader:
                x1, x2, x3, y = x1.to(DEVICE), x2.to(DEVICE), x3.to(DEVICE), y.to(DEVICE)
                logits = model(x1, x2, x3)
                val_preds.extend(logits.argmax(1).cpu().numpy())
                val_true.extend(y.cpu().numpy())
        
        val_f1 = f1_score(val_true, val_preds, average='macro')
        val_acc = accuracy_score(val_true, val_preds)
        
        print(f"Epoch {epoch+1:02d} | Loss: {train_loss/len(train_loader):.4f} | "
              f"Tr F1: {train_f1:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}")
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), "platinum_model.pth")
            print(f"  >>> New Best F1: {best_f1:.4f}")

    print(f"🏆 Final Best Val F1: {best_f1}")
    return model

# 6. EXECUTE PIPELINE
model = train_and_eval()

# 7. GENERATE SUBMISSION
print("\n📝 Generating Submission CSV...")
test_ds = TripleImageDataset(DATA_PATH, split="test", transform=val_transform)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

model.load_state_dict(torch.load("platinum_model.pth"))
model.eval()

ids, preds = [], []
with torch.no_grad():
    for (x1, x2, x3), img_id in test_dl:
        x1, x2, x3 = x1.to(DEVICE), x2.to(DEVICE), x3.to(DEVICE)
        out = model(x1, x2, x3)
        preds.extend(out.argmax(1).cpu().numpy())
        ids.extend(img_id.numpy())

df = pd.DataFrame({'id': ids, 'target': preds})
df.to_csv("submission_platinum.csv", index=False)
print("✅ submission_platinum.csv created.")

# 8. GENERATE METRICS & CONFUSION MATRIX
print("\n📊 Generating Metrics & Heatmap...")
y_true, y_pred = [], []
with torch.no_grad():
    for (x1, x2, x3), targets in val_loader:
        x1, x2, x3 = x1.to(DEVICE), x2.to(DEVICE), x3.to(DEVICE)
        outputs = model(x1, x2, x3)
        y_true.extend(targets.cpu().numpy())
        y_pred.extend(outputs.argmax(1).cpu().numpy())

# Text Report
report = classification_report(y_true, y_pred, digits=4)
with open("metrics_platinum.txt", "w") as f: f.write(report)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Platinum Model')
plt.savefig("confusion_matrix_platinum.png")
plt.close()
print("✅ Metrics and Plot saved.")

# 9. SAFE DOWNLOAD FUNCTION
def package_and_download_full():
    files_to_zip = ["platinum_model.pth", "submission_platinum.csv", "metrics_platinum.txt", "confusion_matrix_platinum.png"]
    zip_name = "Platinum_Full_Package.zip"
    
    with zipfile.ZipFile(zip_name, 'w') as zipf:
        found_count = 0
        for f in files_to_zip:
            if os.path.exists(f): 
                zipf.write(f, arcname=os.path.basename(f))
                found_count += 1
                print(f"📦 Added to zip: {f}")
            else:
                print(f"⚠️ Missing: {f}")
    
    if found_count > 0:
        b64 = base64.b64encode(open(zip_name, 'rb').read()).decode()
        display(Javascript(f"""
        var link = document.createElement('a');
        link.href = "data:application/zip;base64,{b64}";
        link.download = "{zip_name}";
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
        """))
        display(HTML(f'<br><h3><a download="{zip_name}" href="data:application/zip;base64,{b64}">📥 Download Full Package ({zip_name})</a></h3>'))
    else:
        print("❌ Error: No files found to zip.")

package_and_download_full()

✅ Method 1 (Platinum Full Pipeline) running on: cuda
📊 Data Split: 18552 Training | 3273 Validation
🧠 Platinum Model Parameters: 497,752 (Limit: 500,000)
🚀 Starting Platinum Training...
Epoch 01 | Loss: 2.2126 | Tr F1: 0.3147 | Val F1: 0.4208 | Val Acc: 0.4449
  >>> New Best F1: 0.4208
Epoch 02 | Loss: 1.9061 | Tr F1: 0.4483 | Val F1: 0.4641 | Val Acc: 0.5127
  >>> New Best F1: 0.4641
Epoch 03 | Loss: 1.7850 | Tr F1: 0.4993 | Val F1: 0.5351 | Val Acc: 0.5713
  >>> New Best F1: 0.5351
Epoch 04 | Loss: 1.6804 | Tr F1: 0.5498 | Val F1: 0.5548 | Val Acc: 0.5888
  >>> New Best F1: 0.5548
Epoch 05 | Loss: 1.5976 | Tr F1: 0.5860 | Val F1: 0.6049 | Val Acc: 0.6324
  >>> New Best F1: 0.6049
Epoch 06 | Loss: 1.5334 | Tr F1: 0.6184 | Val F1: 0.6619 | Val Acc: 0.6783
  >>> New Best F1: 0.6619
Epoch 07 | Loss: 1.4838 | Tr F1: 0.6433 | Val F1: 0.6516 | Val Acc: 0.6685
Epoch 08 | Loss: 1.4305 | Tr F1: 0.6656 | Val F1: 0.6582 | Val Acc: 0.6835
Epoch 09 | Loss: 1.3855 | Tr F1: 0.6900 | Val F1: 0.6961 |

<IPython.core.display.Javascript object>